**installation pyspark**

In [ ]:
!pip install -q pyspark

In [23]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TempView Example") \
  .master("local[*]") \
  .getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True) \
  .csv("/content/fifaworldcup.csv")

df.createOrReplaceTempView("Matches")

In [24]:
result = spark.sql("SELECT * FROM Matches")
result.show(10)

+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|      date|home_team|away_team|home_score|away_score|tournament|   city| country|neutral|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|1872-11-30| Scotland|  England|         0|         0|  Friendly|Glasgow|Scotland|  false|
|1873-03-08|  England| Scotland|         4|         2|  Friendly| London| England|  false|
|1874-03-07| Scotland|  England|         2|         1|  Friendly|Glasgow|Scotland|  false|
|1875-03-06|  England| Scotland|         2|         2|  Friendly| London| England|  false|
|1876-03-04| Scotland|  England|         3|         0|  Friendly|Glasgow|Scotland|  false|
|1876-03-25| Scotland|    Wales|         4|         0|  Friendly|Glasgow|Scotland|  false|
|1877-03-03|  England| Scotland|         1|         3|  Friendly| London| England|  false|
|1877-03-05|    Wales| Scotland|         0|         2|  Friendly|Wrexham|   Wales|  false|

In [25]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: string (nullable = true)
 |-- away_score: string (nullable = true)
 |-- tournament: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- neutral: boolean (nullable = true)



In [69]:
df.show()
df.select("date","home_team","away_team","home_score","away_score","tournament").show()

+----------+----------------+---------+----------+----------+----------+---------+--------+-------+
|      date|       home_team|away_team|home_score|away_score|tournament|     city| country|neutral|
+----------+----------------+---------+----------+----------+----------+---------+--------+-------+
|1872-11-30|        Scotland|  England|         0|         0|  Friendly|  Glasgow|Scotland|  false|
|1873-03-08|         England| Scotland|         4|         2|  Friendly|   London| England|  false|
|1874-03-07|        Scotland|  England|         2|         1|  Friendly|  Glasgow|Scotland|  false|
|1875-03-06|         England| Scotland|         2|         2|  Friendly|   London| England|  false|
|1876-03-04|        Scotland|  England|         3|         0|  Friendly|  Glasgow|Scotland|  false|
|1876-03-25|        Scotland|    Wales|         4|         0|  Friendly|  Glasgow|Scotland|  false|
|1877-03-03|         England| Scotland|         1|         3|  Friendly|   London| England|  false|


**Maroc (10 derniers matchs)**

In [70]:
spark.sql("""
SELECT date, home_team, away_team, home_score, away_score, tournament
FROM matches
WHERE (home_team='Morocco' OR away_team='Morocco')
  AND home_score IS NOT NULL AND away_score IS NOT NULL
ORDER BY date DESC
""").show()

+----------+-------------+-------------+----------+----------+--------------------+
|      date|    home_team|    away_team|home_score|away_score|          tournament|
+----------+-------------+-------------+----------+----------+--------------------+
|2022-09-27|     Paraguay|      Morocco|         0|         0|            Friendly|
|2022-09-23|      Morocco|        Chile|         2|         0|            Friendly|
|2022-06-13|      Morocco|      Liberia|         2|         0|African Cup of Na...|
|2022-06-09|      Morocco| South Africa|         2|         1|African Cup of Na...|
|2022-06-01|United States|      Morocco|         3|         0|            Friendly|
|2022-03-29|      Morocco|     DR Congo|         4|         1|FIFA World Cup qu...|
|2022-03-25|     DR Congo|      Morocco|         1|         1|FIFA World Cup qu...|
|2022-01-30|        Egypt|      Morocco|         2|         1|African Cup of Na...|
|2022-01-25|      Morocco|       Malawi|         2|         1|African Cup of

**Stats Maroc (matchs / victoires / nuls / défaites)**

In [65]:
spark.sql("""
WITH maroc AS (
  SELECT
    CASE WHEN home_team='Morocco' THEN home_score ELSE away_score END bm,
    CASE WHEN home_team='Morocco' THEN away_score ELSE home_score END ba
  FROM matches
  WHERE home_team='Morocco' OR away_team='Morocco'
)
SELECT
  COUNT(*) matchs,
  SUM(CASE WHEN bm>ba THEN 1 ELSE 0 END) victoires,
  SUM(CASE WHEN bm=ba THEN 1 ELSE 0 END) nuls,
  SUM(CASE WHEN bm<ba THEN 1 ELSE 0 END) defaites
FROM maroc
WHERE bm IS NOT NULL AND ba IS NOT NULL
""").show()

+------+---------+----+--------+
|matchs|victoires|nuls|defaites|
+------+---------+----+--------+
|   572|      271| 164|     137|
+------+---------+----+--------+



**Buts pour contre du Maroc**

In [66]:
spark.sql("""
WITH maroc AS (
  SELECT
    CASE WHEN home_team='Morocco' THEN home_score ELSE away_score END bp,
    CASE WHEN home_team='Morocco' THEN away_score ELSE home_score END bc
  FROM matches
  WHERE home_team='Morocco' OR away_team='Morocco'
)
SELECT SUM(bp) buts_pour, SUM(bc) buts_contre, SUM(bp)-SUM(bc) diff
FROM maroc
WHERE bp IS NOT NULL AND bc IS NOT NULL
""").show()

+---------+-----------+----+
|buts_pour|buts_contre|diff|
+---------+-----------+----+
|      829|        481| 348|
+---------+-----------+----+

